In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path("..").resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RAW_DIR = PROJECT_DIR / "data" / "raw"

hourly = pd.read_csv(PROCESSED_DIR / "hourly_pickups.csv", parse_dates=["hour"])
clusters = pd.read_csv(PROCESSED_DIR / "station_clusters.csv")
weather = pd.read_csv(PROCESSED_DIR / "weather_hourly.csv", parse_dates=["hour"])

print(f"Hourly pickups: {hourly.shape}")
print(f"Station clusters: {clusters.shape}")
print(f"Weather: {weather.shape}")

Hourly pickups: (10705152, 3)
Station clusters: (1031, 3)
Weather: (10176, 5)


## aggregating to cluster level

In [ ]:
df = hourly.merge(clusters[["station_id", "cluster_id"]], on="station_id", how="inner")
cluster_agg = df.groupby(["cluster_id", "hour"])["pickups"].sum().reset_index()

cluster_cap = clusters.drop_duplicates("cluster_id")[["cluster_id", "cluster_capacity"]]
cluster_agg = cluster_agg.merge(cluster_cap, on="cluster_id")

cluster_agg = cluster_agg.sort_values(["cluster_id", "hour"])
cluster_agg["lag_24h"] = cluster_agg.groupby("cluster_id")["pickups"].shift(24)  # adding lag feature for same hour yesterday
cluster_agg = cluster_agg.dropna(subset=["lag_24h"])  # drops first 24h per cluster

print(f"Cluster-hour rows: {len(cluster_agg):,}")
print(f"Clusters: {cluster_agg['cluster_id'].nunique()}, Hours: {cluster_agg['hour'].nunique()}")
print(f"Date range: {cluster_agg['hour'].min()} to {cluster_agg['hour'].max()}")

Cluster-hour rows: 81,216
Clusters: 8, Hours: 10152
Date range: 2025-01-02 00:00:00 to 2026-02-28 23:00:00


## merging weather

In [3]:
cluster_agg = cluster_agg.merge(weather, on="hour", how="left")
print(f"After weather merge: {cluster_agg.shape}")
print(f"Weather nulls: {cluster_agg[['temperature_c','precipitation_mm','wind_speed_kmh']].isnull().sum().to_dict()}")

After weather merge: (81216, 9)
Weather nulls: {'temperature_c': 0, 'precipitation_mm': 0, 'wind_speed_kmh': 0}


## temporal and weather features

In [4]:
HOLIDAYS = {
    "2025-01-01", "2025-02-17", "2025-04-18", "2025-05-19",
    "2025-07-01", "2025-08-04", "2025-09-01", "2025-10-13",
    "2025-12-25", "2025-12-26",
    "2026-01-01", "2026-02-16",
}

df = cluster_agg.copy()
df["hour_of_day"] = df["hour"].dt.hour
df["day_of_week"] = df["hour"].dt.dayofweek  # 0=Monday
df["month"] = df["hour"].dt.month
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
df["is_rush_hour"] = (
    (df["hour_of_day"].between(7, 9) | df["hour_of_day"].between(16, 18)) &
    (df["is_weekend"] == 0)
).astype(int)
df["is_holiday"] = df["hour"].dt.date.astype(str).isin(HOLIDAYS).astype(int)
df["is_rainy"] = (df["precipitation_mm"] > 0).astype(int)

FEATURES = [
    "cluster_capacity",
    "hour_of_day", "day_of_week", "month", "is_weekend", "is_rush_hour", "is_holiday",
    "temperature_c", "precipitation_mm", "wind_speed_kmh", "is_rainy",
    "lag_24h",
]
print(f"Features ({len(FEATURES)}): {FEATURES}")
print(df[FEATURES + ["pickups"]].describe().round(2))

Features (12): ['cluster_capacity', 'hour_of_day', 'day_of_week', 'month', 'is_weekend', 'is_rush_hour', 'is_holiday', 'temperature_c', 'precipitation_mm', 'wind_speed_kmh', 'is_rainy', 'lag_24h']
       cluster_capacity  hour_of_day  day_of_week     month  is_weekend  \
count          81216.00     81216.00     81216.00  81216.00    81216.00   
mean            2454.00        11.50         3.01      5.83        0.29   
std             2021.75         6.92         2.00      3.65        0.45   
min              643.00         0.00         0.00      1.00        0.00   
25%             1033.00         5.75         1.00      2.00        0.00   
50%             1795.50        11.50         3.00      6.00        0.00   
75%             2694.75        17.25         5.00      9.00        1.00   
max             7163.00        23.00         6.00     12.00        1.00   

       is_rush_hour  is_holiday  temperature_c  precipitation_mm  \
count      81216.00    81216.00       81216.00          812

## train / val / test split

In [5]:
train = df[df["hour"].dt.year == 2025].copy()
val   = df[(df["hour"].dt.year == 2026) & (df["hour"].dt.month == 1)].copy()
test  = df[(df["hour"].dt.year == 2026) & (df["hour"].dt.month == 2)].copy()

print(f"Train: {train.shape}  ({train['hour'].min().date()} to {train['hour'].max().date()})")
print(f"Val:   {val.shape}  ({val['hour'].min().date()} to {val['hour'].max().date()})")
print(f"Test:  {test.shape}  ({test['hour'].min().date()} to {test['hour'].max().date()})")
print(f"\nTarget stats:")
print(f"  train  mean={train['pickups'].mean():.1f}  std={train['pickups'].std():.1f}")
print(f"  val    mean={val['pickups'].mean():.1f}  std={val['pickups'].std():.1f}")
print(f"  test   mean={test['pickups'].mean():.1f}  std={test['pickups'].std():.1f}")

Train: (69888, 16)  (2025-01-02 to 2025-12-31)
Val:   (5952, 16)  (2026-01-01 to 2026-01-31)
Test:  (5376, 16)  (2026-02-01 to 2026-02-28)

Target stats:
  train  mean=96.0  std=282.7
  val    mean=22.7  std=77.5
  test   mean=20.7  std=61.1


## saving

In [6]:
KEEP_COLS = ["cluster_id", "hour", "pickups"] + FEATURES

train[KEEP_COLS].to_csv(PROCESSED_DIR / "train.csv", index=False)
val[KEEP_COLS].to_csv(PROCESSED_DIR / "val.csv", index=False)
test[KEEP_COLS].to_csv(PROCESSED_DIR / "test.csv", index=False)

print("Saved train.csv, val.csv, test.csv to data/processed/")
print(f"Columns ({len(KEEP_COLS)}): {KEEP_COLS}")

Saved train.csv, val.csv, test.csv to data/processed/
Columns (15): ['cluster_id', 'hour', 'pickups', 'cluster_capacity', 'hour_of_day', 'day_of_week', 'month', 'is_weekend', 'is_rush_hour', 'is_holiday', 'temperature_c', 'precipitation_mm', 'wind_speed_kmh', 'is_rainy', 'lag_24h']
